In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient IV: how far may one update move?

## Where this session fits

> **The limitation we start from: every episode is expensive, and the policy gradient uses a batch of episodes for *one* update and then throws it away**

- **a second problem:** a large step on a noisy estimate can also ruin the policy

- **The question:** how can we take several updates from the same batch, and keep each one small enough to trust?
- **Position: Part B (policy gradient), session 4 of its 4 sessions (the last of Part B)**
    - B1 gave the theorem
    - B2 weight and B3 critic filled slot 1, the weight on each step (an advantage)
    - this session fills slot 2: *how far may one update move?*
- **The three slots** of the plan from B1 (theorem): the parts of the gradient formula that the methods change
    1. **slot 1, the weight** $w$ on each gradient term
    2. **slot 2, how far** one update moves the policy **← this session**
    3. **slot 3, where the baseline comes from** (inside slot 1)
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say why a batch of episodes goes stale after one update, and what the probability ratio corrects
2. say what a surrogate objective is (a stand-in for the true goal) and why its gradient at ratio $1$ is the policy gradient
3. read the clipped objective, and say which samples it stops pushing
4. read the PPO outline and say which model does what
5. say how a trust region and clipping limit the movement, and what a KL penalty adds (developed in Part C)


**Where this session sits in Part B**

<img src="images/partB_arc_b4.png" width=95%>


# The limitation: a batch goes stale after one update

**Why this section?** B2 made the weight good. The next cost is how much each batch of episodes buys us.

> **A batch of episodes goes stale after one update. The fix has two parts: correct for data from the old policy (the probability ratio), and limit how far $\theta$ moves.**

- the policy gradient is **on-policy**: the episodes must come from the policy being improved (the idea from sessions A3 and A4)
    - the expectation in the theorem is over $\traj \sim \pi_\theta$
- one update changes $\theta$, so the old episodes now came from a *different* policy, the **old policy** $\pi_{\theta_{\text{old}}}$
    - REINFORCE and actor-critic (B2 weight and B3 critic) sample, update once, and discard

**Two costs, and the fix**

- two costs follow
    - **waste:** generating episodes is the expensive part, especially for language models
        - a language-model episode is one response, generated one token (one step) at a time
        - so the number of steps equals the length of the response, which may be long
        - one update per batch uses little of what we paid for
    - **risk:** a bigger step would use the batch better, but the gradient is a *noisy local estimate*
        - far from $\theta_{\text{old}}$ it may point the wrong way, and a policy that collapses produces worse data for the next update
- the fix has two parts
    - **correct** for the fact that the data came from the old policy: the probability ratio (next slide)
    - **limit** how far $\theta$ moves from $\theta_{\text{old}}$
        - we optimize a **surrogate**: a stand-in for the true objective, defined a few slides on
        - and constrain it, so that $\theta$ stays where the stand-in is trustworthy


## Reusing old data: the probability ratio

**Why this section?** This is the on-policy versus off-policy thread, now for a policy gradient: data from one policy, an expectation under another.

> **The probability ratio $r = \pi_\theta / \pi_{\theta_{\text{old}}}$ compares the new policy with the old one, and reweights old-policy data to give the new policy's average.**

- the **probability ratio** compares the new policy with the old one, for one state $\state$ and one action $\act$
    - $r_\theta(\state, \act) = \dfrac{\pi_\theta(\act \mid \state)}{\pi_{\theta_{\text{old}}}(\act \mid \state)}$
    - it depends on three things: the parameters $\theta$, the state, and the action
    - $r > 1$: the action became more likely
    - $r < 1$: less likely
    - $r = 1$ at $\theta = \theta_{\text{old}}$
- **two views of the same ratio**, depending on what varies
    - **training:** the batch fixes the step's state and action, so we write $r_\tt(\theta) = r_\theta(\state_\tt, \act_\tt)$ and vary only $\theta$
    - **the identity below:** $\theta$ is fixed and the action varies
- as an **importance weight**: averaging $f(\act)$ over actions *sampled by the old policy* and multiplying by $r$ gives the average under the *new* policy
    - $\E_{\act \sim \pi_{\theta_{\text{old}}}(\cdot \mid \state)} \bigl[\, r_\theta(\state, \act)\, f(\act) \,\bigr] = \E_{\act \sim \pi_\theta(\cdot \mid \state)} \bigl[\, f(\act) \,\bigr]$
    - here $\act$ is an action; do not read it as the advantage $\advseq$

**The ratio by numbers**

> **Goal: the average advantage under the *new* policy, computed from actions sampled by the *old* policy.**

One state, three actions (synthetic numbers):

| action | old policy $\pi_{\theta_{\text{old}}}$ | new policy $\pi_\theta$ | ratio $r = \text{new} / \text{old}$ | advantage $\advseq$ |
|---|---|---|---|---|
| 0 | 0.5 | 0.6 | 1.2 | $+1$ |
| 1 | 0.3 | 0.25 | 0.833 | $-1$ |
| 2 | 0.2 | 0.15 | 0.75 | $-1$ |

- the advantage of each action is measured against the old policy, so its old-policy average is zero
    - $0.5 \cdot (+1) + 0.3 \cdot (-1) + 0.2 \cdot (-1) = 0$
- **directly**, weighting by the new policy: $0.6 \cdot (+1) + 0.25 \cdot (-1) + 0.15 \cdot (-1) = 0.20$
- **from old samples**, weighting each by $r$: $0.5 \cdot 1.2 \cdot (+1) + 0.3 \cdot 0.833 \cdot (-1) + 0.2 \cdot 0.75 \cdot (-1) = 0.20$
    - the same answer: $r$ turns the old probability into the new one
    - **0.20 is the surrogate's improvement** of the new policy over the old one, in this state
- the identity needs the old policy to give positive probability to every action the new policy can take

<img src="images/b3_ratio_bars.png" width=55%>

- notation: this $r$ is the PPO paper's ratio, not a reward (written $\rew$ here)
    - context separates them


## The surrogate objective

**Why this section?** An objective we can optimize more than once on the same batch, whose gradient is still the policy gradient where it matters.

> **New term. A *surrogate* is a stand-in for the true goal $J(\theta)$, built from old data. It tracks the improvement in $J$ near $\theta_{\text{old}}$ and is trusted only there.**

- the **surrogate objective**: the batch average of ratio times advantage
    - $J_{\text{sur}}(\theta) = \E_\tt \bigl[\, r_\tt(\theta) \, \advseq_\tt \,\bigr]$
    - $\E_\tt$ is the average over the time steps of the batch
    - the advantage comes from B2 (weight), computed with the old policy
- like $J$, it is **maximized**: a higher value means a better policy
- why a stand-in?
    - $J(\theta)$ can be estimated only from fresh episodes of the current policy
    - the surrogate can be evaluated for many values of $\theta$ from one old batch, because the ratio corrects for the old data
- what it estimates: the **improvement** $J(\theta) - J(\theta_{\text{old}})$
    - at $\theta = \theta_{\text{old}}$ it is $0$: the ratio is $1$ and the advantages average to zero (in expectation)
    - its gradient there is the policy gradient of B2 (weight), in the form we use (no $\disc^\tt$ factor)
- how far to trust it is a separate question: the next slide measures it, and the section after keeps $\theta$ near $\theta_{\text{old}}$


**How good a stand-in? What is exact and what is approximate**

> **The surrogate gets the new policy's actions right but the states wrong: exact at $\theta_{\text{old}}$, with an error that grows, then levels off, as $\theta$ moves away.**

- the exact identity (the performance difference lemma, Kakade and Langford 2002)
    - true improvement = the average of the *old* advantages
    - taken over the *new* policy's actions **and the states the new policy visits**
- the surrogate keeps the new policy's actions (through $r$) but averages over the states the *old* policy visited
    - so it is exact when both policies visit the same states, as at $\theta = \theta_{\text{old}}$
    - the whole error comes from the change in which states are visited
- TRPO (Schulman et al. 2015) proves a bound: the error is limited by a constant times the **KL divergence** between old and new policy
    - the KL divergence measures how different two distributions are: zero when they agree
- **measured** (synthetic: 40 random small MDPs, exact $J$, moving $\theta$ along the policy gradient)
    - the median error is 2% of the true improvement at an average KL of 0.01, 7% at 0.05, and 9% at 0.1
    - it varies a lot between MDPs: at 0.1 the middle 80% range from 2% to 22%


**The measured error** (synthetic: 40 random small MDPs)

<img src="images/b4_surrogate_accuracy.png" width=95%>

- the figure: one example MDP (left), then the error for each of the 40 MDPs (right)
    - the median is in orange
    - left: the horizontal axis is the step along the gradient direction, in arbitrary units
- the figure uses discounted state weights, which the batch average here drops (as in B2 (weight))


**Why the gradient matches, and a sign convention**

- why the gradient matches: differentiate the ratio
    - only the numerator of $r_\tt$ depends on $\theta$, so $\pi_{\theta_{\text{old}}}$ stays outside the gradient
    - $\nabla_\theta r_\tt(\theta) = \dfrac{\nabla_\theta \pi_\theta(\actseq_\tt \mid \stateseq_\tt)}{\pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)} = r_\tt(\theta) \, \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt)$
    - the second step uses $\nabla \pi = \pi \, \nabla \log \pi$ and $\pi_\theta / \pi_{\theta_{\text{old}}} = r_\tt$
    - at $r_\tt = 1$ this is $\nabla_\theta \log \pi_\theta \; \advseq_\tt$, the term of the unified form
- **caution** (measured on the previous slide): the ratio corrects the choice of *action* in each visited state, not *which states are visited*
    - that is why we stay near $\theta_{\text{old}}$
- **sign convention:** the surrogate is *maximized*, like $J$: a surrogate **objective**, as in the PPO and GRPO papers
    - code usually minimizes, so it minimizes the negated surrogate, $-J_{\text{sur}}$
    - that negated version is often called the **surrogate loss**


## Keeping $\theta$ near $\theta_{\text{old}}$: three mechanisms

**Why this section?** The surrogate is trustworthy only near $\theta_{\text{old}}$. Methods differ in how they enforce that.

> **Three ways to keep $\theta$ near $\theta_{\text{old}}$: a trust region, clipping, and a KL penalty.**

| Mechanism | How | Used by |
|:--|:--|:--|
| **trust region** | a hard bound on the KL divergence between old and new policy (a measure of how different two distributions are) | TRPO (this session) |
| **clipping** | a sample stops pushing once its ratio has moved past $1 \pm \epsilon_{\text{clip}}$, in the direction its advantage wants (below) | PPO (this session) |
| **KL penalty** | a penalty on drifting away from another policy: the old one, or a frozen reference model | some PPO variants; RLHF-style PPO and GRPO (Part C); RLHF is reinforcement learning from human feedback |

- clipping, in words: a sample stops pushing once the policy has already moved enough on it
    - a good action ($\advseq > 0$) stops once $r > 1 + \epsilon_{\text{clip}}$: it is already much more likely
    - a bad action ($\advseq < 0$) stops once $r < 1 - \epsilon_{\text{clip}}$: it is already much less likely
    - "stops" means its gradient is exactly zero there; the next slides show it

- all three limit **how far the policy moves**; only clipping works through the ratio
    - the trust region and the KL penalty act on the KL divergence between whole distributions
    - clipping acts on the ratio of the *sampled* action, a cheaper stand-in
    - the previous slide measured the surrogate's error against KL, so the KL-based limits are the ones it directly supports


**Within an iteration, or anchored to another policy**

- the first two limit the movement *within an iteration*, relative to the **old policy** $\pi_{\theta_{\text{old}}}$ that made the batch
    - the old policy is replaced at every iteration
- the third can instead anchor to a different, fixed policy: the **reference model** $\pi_{\text{ref}}$ (also called the reference policy)
    - a frozen copy of a model we do not want to leave at all, fixed for the whole run
    - for a language model, usually the supervised fine-tuned model
- it adds $-\beta \, \KL(\cdot)$ to the objective, with a weight $\beta$ (Part C)
- $\KL(p \,\|\, q)$ is the average, over samples from $p$, of $\log p - \log q$: zero when the two agree, larger as they differ, not symmetric
- PPO is the practical, widely used choice: no second-order machinery (that is, no second derivatives of the objective)


# PPO: clip the ratio

**Why this section?** The clipped objective is the whole idea of PPO (proximal policy optimization): stop pushing a sample once its ratio has moved far enough.

> **PPO clips the ratio: once a sample's ratio has moved far enough in the direction its advantage favors, that sample is switched off: its gradient is exactly zero.**

- the **clipped surrogate** (maximized)

$$
J_{\text{clip}}(\theta) = \E_\tt \Bigl[\, \min{} \bigl( r_\tt(\theta)\, \advseq_\tt, \;\; \text{clip}\bigl(r_\tt(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \advseq_\tt \bigr) \,\Bigr]
$$

- $\text{clip}(x, a, b)$ limits $x$ to the interval $[a, b]$
- $\epsilon_{\text{clip}}$ is the clip range, typically $0.2$
- the paper calls the range $\epsilon$
- we add the subscript to avoid the exploration probability $\epsilon$ of $\epsilon$-greedy

**What clipping does**

> **Clipping does not merely cap the ratio: past the range it replaces the ratio by a constant. The ratio drops out of that sample's term, so its gradient is exactly $0$.**
>
> In that case the `min` picks the clipped term $(1 \pm \epsilon_{\text{clip}})\,\advseq$, which contains no $\theta$.

<img src="images/ppo_clip.png" width=80%>

- with a positive advantage the objective stops rewarding an increase of $r$ beyond $1 + \epsilon_{\text{clip}}$
- with a negative advantage, a decrease below $1 - \epsilon_{\text{clip}}$
- where the curve is flat, the gradient is zero: that sample no longer pushes the policy
    - a **hard cutoff**, in one direction only: not the gradual fading of a "vanishing gradient" in a deep network
- the sloped branches keep the gradient, so a mistake (a ratio that moved the wrong way) can be undone


**What clipping does not do**

- clipping does **not** bound the policy: other samples share the parameters and can still push a clipped sample's ratio further, so PPO has no guarantee
- clipping does **not** force $r$ to stay in the range
- it removes the gradient of a sample once its ratio has moved *past* the range in the direction its advantage favors
- the `min` keeps the pessimistic one
    - it never rewards a sample for moving further than the range allows
    - but it still pushes back a ratio that moved the *wrong* way


**Four samples, by numbers** ($\epsilon_{\text{clip}} = 0.2$, so the range is $[0.8, 1.2]$)

<table>
<thead><tr><th nowrap><div align="left">Advantage</div></th><th nowrap><div align="left">Ratio $r$</div></th><th nowrap><div align="left">$r \, \advseq$</div></th><th nowrap><div align="left">clipped term</div></th><th nowrap><div align="left"><code>min</code></div></th><th nowrap><div align="left">Gradient</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">$+2$</div></td><td nowrap><div align="left">$1.3$</div></td><td nowrap><div align="left">$2.6$</div></td><td nowrap><div align="left">$1.2 \cdot 2 = 2.4$</div></td><td nowrap><div align="left">$2.4$ (clipped)</div></td><td nowrap><div align="left"><b>zero</b>: already pushed up enough</div></td></tr>
<tr><td nowrap><div align="left">$+2$</div></td><td nowrap><div align="left">$0.7$</div></td><td nowrap><div align="left">$1.4$</div></td><td nowrap><div align="left">$0.8 \cdot 2 = 1.6$</div></td><td nowrap><div align="left">$1.4$ (unclipped)</div></td><td nowrap><div align="left"><b>flows</b>: pushes the ratio back up</div></td></tr>
<tr><td nowrap><div align="left">$-1$</div></td><td nowrap><div align="left">$1.3$</div></td><td nowrap><div align="left">$-1.3$</div></td><td nowrap><div align="left">$1.2 \cdot (-1) = -1.2$</div></td><td nowrap><div align="left">$-1.3$ (unclipped)</div></td><td nowrap><div align="left"><b>flows</b>: pushes the ratio back down</div></td></tr>
<tr><td nowrap><div align="left">$-1$</div></td><td nowrap><div align="left">$0.7$</div></td><td nowrap><div align="left">$-0.7$</div></td><td nowrap><div align="left">$0.8 \cdot (-1) = -0.8$</div></td><td nowrap><div align="left">$-0.8$ (clipped)</div></td><td nowrap><div align="left"><b>zero</b>: already pushed down enough</div></td></tr>
</tbody></table>

- samples in the first and fourth rows have moved far enough in the direction their advantage asks
- samples in the second and third rows moved the wrong way, so the gradient still acts
- inside the range $[0.8, 1.2]$ nothing is clipped and the surrogate is the plain ratio objective


## Iterations and epochs: reusing one batch

**Why this section?** Reuse is the goal of the whole session. The words must be exact, because supervised learning uses them differently.

> **An iteration generates a batch with the current policy; epochs are several passes over that fixed batch.**

<img src="images/b4_iterations_epochs.png" width=85%>

- in supervised learning, an **epoch** is one pass over fixed examples
- here the examples are episodes made by the *current* policy, so there are two nested loops
    - an **iteration** generates a batch with the policy as it is at the start, which becomes the old policy $\pi_{\theta_{\text{old}}}$
    - inside it, **epochs**: several passes over that *fixed* batch, updating the actor (and the critic) each time
    - at the end of the iteration $\theta_{\text{old}}$ is set to the new $\theta$

**Reusing a batch: is it safe?**

- why this is safe enough
    - the first update is exactly on-policy: $r_\tt = 1$ and the gradient is the policy gradient
        - implementations split a batch into mini-batches, so later mini-batches already have $r \neq 1$
    - later epochs are slightly off-policy
    - the ratio corrects them, and clipping stops the samples that have moved far
- the number of epochs $K$ (often $3$ to $10$) is the dial
    - more epochs use the batch better
    - but the samples drift further from the old policy


## PPO in one loop

**Why this section?** Everything so far as one algorithm. The comments say which model computes what.

> **PPO in one loop: roll out, freeze, then several epochs of clipped updates.**

- **models:** actor $\pi_\theta$, critic $\statevalfun_\phi$, old actor $\pi_{\text{old}}$ (a frozen snapshot of the actor)
- for each iteration
    - **rollout:** collect actions, states and rewards with the current actor
        - set $\pi_{\text{old}} \leftarrow \pi_\theta$ (it produced the batch and is not updated below)
    - **frozen for the whole iteration (no gradient):** $\log \pi_{\text{old}}(\actseq_\tt \mid \stateseq_\tt)$ of each sampled step
        - the values $\statevalfun_\phi(\stateseq_\tt)$
        - the advantages $\hat{\advseq}_\tt$ by GAE from the rewards and values (B3 critic, appendix)
        - the critic's targets $\hat{\advseq}_\tt + \statevalfun_\phi(\stateseq_\tt)$ (a terminal state has value $0$)
    - for epoch $= 1, \ldots, K$ (full-batch passes here; implementations split the batch into mini-batches)
        - ratio $r_\tt = \exp\bigl(\log \pi_\theta(\actseq_\tt \mid \stateseq_\tt) - \log \pi_{\text{old}}(\actseq_\tt \mid \stateseq_\tt)\bigr)$ (same actions, current weights)
        - actor loss: minus the mean over $\tt$ of $\min{}\bigl(r_\tt \hat{\advseq}_\tt, \; \text{clip}(r_\tt, 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}) \hat{\advseq}_\tt\bigr)$ (minus: we minimize)
        - critic loss: the mean over $\tt$ of the squared difference between $\statevalfun_\phi(\stateseq_\tt)$ and its target
        - update $\theta$ and $\phi$ to reduce the actor loss plus $c$ times the critic loss
            - $c$ weights the critic loss
            - practice often adds an *entropy bonus*: a term that rewards keeping the policy random, to preserve exploration


**Notes on the loop**

- the actor is updated every epoch
- $\pi_{\text{old}}$ changes only between iterations
- the advantage is computed once per iteration, with the old critic, and held fixed
- *RLHF-style PPO adds a reference model and a KL term to this loop*
    - *RLHF: reinforcement learning from human feedback, that is, RL on a language model with a reward learned from human judgments*
    - *C1 gives the details*


**Which model does what**

<table>
<thead><tr><th nowrap><div align="left">Model</div></th><th nowrap><div align="left">Role</div></th><th nowrap><div align="left">Updated</div></th><th nowrap><div align="left">Needed for</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">actor $\pi_\theta$</div></td><td nowrap><div align="left">the policy we are improving</div></td><td nowrap><div align="left">every epoch</div></td><td nowrap><div align="left">the ratio's numerator, and the next iteration's data</div></td></tr>
<tr><td nowrap><div align="left">old actor $\pi_{\text{old}}$</div></td><td nowrap><div align="left">snapshot that made the batch</div></td><td nowrap><div align="left">between iterations</div></td><td nowrap><div align="left">the ratio's denominator ($\log \pi_{\text{old}}$ stored once)</div></td></tr>
<tr><td nowrap><div align="left">critic $\statevalfun_\phi$</div></td><td nowrap><div align="left">baseline source (slot 3) and the advantage via GAE</div></td><td nowrap><div align="left">every epoch, by squared error</div></td><td nowrap><div align="left">the advantage estimate</div></td></tr>
</tbody></table>

- in practice the old actor is often only its stored log-probabilities, not a second network held in memory
- the critic is a **second set of parameters** to train
- it may share layers with the actor, but for large language models it is typically a separate network of similar size
- that cost motivates the question GRPO answers in Part C


## Where the idea comes from: TRPO

**Why this section?** Clipping is a practical stand-in for a more principled constraint. The history explains why it looks the way it does.

> **TRPO maximizes the surrogate subject to a bound on how far the new policy may move; PPO is a first-order heuristic in its spirit.**

- **TRPO** (trust region policy optimization): maximize the surrogate *subject to* a bound on how far the new policy may move
    - $\displaystyle \max{\theta} \;\E_\tt \bigl[ r_\tt(\theta)\, \advseq_\tt \bigr] \quad \text{subject to} \quad \E_{\state} \bigl[ \KL \bigl( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \bigr) \bigr] \leq \delta_{\text{KL}}$
    - the **trust region**: the set of policies we trust the local surrogate on


**The same trade-off as a penalty: the weight $\beta$**

- the **KL-penalized form**: the surrogate minus a weighted KL
    - $\displaystyle \max{\theta} \;\E_\tt \bigl[ r_\tt(\theta)\, \advseq_\tt \bigr] \;-\; \beta \, \E_{\state} \bigl[ \KL \bigl( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \bigr) \bigr]$
    - $\beta$ is a **hyper-parameter**: the cost of one unit of KL, relative to one unit of surrogate improvement
    - larger $\beta$ gives smaller steps; $\beta = 0$ puts no limit on the step
- the constraint ($\delta_{\text{KL}}$) and the penalty ($\beta$) are two views of one trade-off
    - this is the form of the KL penalty on the earlier slide, there anchored to a reference model
    - the direction differs: here the old policy comes first, $\KL(\pi_{\theta_{\text{old}}} \,\|\, \pi_\theta)$
    - Part C's penalty averages over the current policy: $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$
- TRPO is motivated by a monotonic-improvement bound for the **penalized** form
    - the $\beta$ that the theory asks for is so large that the steps would be tiny
    - so the practical algorithm uses the constraint with a chosen $\delta_{\text{KL}}$, which carries no such guarantee
    - the constraint needs second-order machinery, which is expensive for a large network

**PPO as a stand-in for the constraint**

- **PPO** is a heuristic in the spirit of TRPO
    - clipping the ratio replaces the hard constraint, with no guarantee: the ratio can still leave the range
    - it uses plain first-order gradient steps, so it scales to large networks


## Where PPO sits: four themes that recur in the course

**Why this section?** These themes came up in Parts A and B. Each says what PPO changes and what it keeps.

- **on-policy versus off-policy:** each iteration is on-policy
    - the actor is updated inside the iteration, so after the first update the batch is no longer from the current policy
        - later epochs are slightly off-policy
    - the ratio corrects this for each action (not for the state distribution)
- **bias and variance:** the weight is still the advantage from B2 (the weight)
    - the critic lowers variance at the price of **bias**, and clipping buys stability at the price of some bias (it truncates the gradient)
- **evaluate versus improve:** the critic evaluates and the actor improves, the same loop as in value-based control
- **backup depth:** GAE sets how far the advantage looks ahead (appendix to B3, the critic)

## The slot table: PPO changes only slot 2

**Why this section?** Placing PPO in the grid we have been filling, so the next method is a single change.

- **slots** (the plan from B1 (theorem))
    - slot 1 is the weight (B2 weight and B3 critic)
    - slot 2 is how far one update moves (this session)
    - slot 3 is where the baseline comes from, inside slot 1 (B2 weight and B3 critic, and Part C)

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">raw theorem</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">plain step</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE</div></td><td nowrap><div align="left">return minus baseline</div></td><td nowrap><div align="left">plain step</div></td><td nowrap><div align="left">moving average</div></td></tr>
<tr><td nowrap><div align="left">actor-critic</div></td><td nowrap><div align="left">TD error or GAE</div></td><td nowrap><div align="left">plain step</div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left"><b>PPO</b></div></td><td nowrap><div align="left">GAE with a learned critic</div></td><td nowrap><div align="left"><b>clipped ratio, $K$ epochs per batch</b></div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left">GRPO</div></td><td nowrap><div align="left">group-relative advantage</div></td><td nowrap><div align="left">clipped ratio, as PPO (Part C)</div></td><td nowrap><div align="left">mean of a group of episodes</div></td></tr>
</tbody></table>


## The new limitation: the critic

**Why this section?** PPO's weight comes from a critic. That is a cost of its own.

- **PPO needs a learned critic**
    - a second network to train, store and run (for large language models, it can be as large as the policy)
    - the critic's mistakes become bias in the advantage
    - for a language model with a reward only at the end of a long response, the critic must score every prefix
- the question for Part C: **do we need the critic at all?**
    - sample *several* episodes from the same start and use their *own average* as the baseline
    - slot 3 (where the baseline comes from) changes
    - slot 2 (how far one update moves) keeps the clipped ratio


## Aside: the Kullback-Leibler direction in TRPO

- $\KL(p \,\|\, q) = \E_{x \sim p}\bigl[ \log p(x) - \log q(x) \bigr]$ is not symmetric in $p$ and $q$
- TRPO writes $\pi_{\theta_{\text{old}}}$ first: the expectation is over the *old* policy, the one that generated the data
- an action the new policy introduces but the old one never took has weight $0$ in this expectation, so it has no *direct* effect on the constraint, only an indirect one through the probability it takes from other actions


# What we established, and what it sets up

- **Established**
    - **PPO limits how far one update moves, so a batch can be reused for several epochs**
    - the policy gradient is on-policy, so a batch goes stale after one update
    - reusing it needs a correction and a limit
    - the **probability ratio** is an importance weight: a re-weighting that lets old-policy samples estimate a new-policy average
    - the **surrogate** $\E[r_\tt \advseq_\tt]$ has the policy gradient as its gradient at ratio $1$
    - **PPO** clips the ratio, so samples that have moved far enough in the direction of their advantage stop pushing
        - this allows several epochs per batch
    - **TRPO** pursues the same goal as clipping with a hard KL constraint
        - the penalty toward a frozen reference policy returns with language models (Part C)
    - PPO's remaining cost: a critic, a second large network
- **Sets up**
    - Part C: slot 3, do we need the critic? GRPO uses a group of episodes as the baseline and keeps the clipped ratio
